In [ ]:
import pandas as pd


import plotly.express as px

### **Analise Exploratória**

In [ ]:
df = pd.read_csv('Cafeteria.csv')
df.head()

,ID da Transação,Item,Quantidade,Preço Unitário,Total Gasto,Método de Pagamento,Localização,Data da Transação
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


* Quantidade de linhas e coluna

In [ ]:
df.shape

(10000, 8)

* Verificar tipos de dados e colunas

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ID da Transação      10000 non-null  object
 1   Item                 9667 non-null   object
 2   Quantidade           9862 non-null   object
 3   Preço Unitário       9821 non-null   object
 4   Total Gasto          9827 non-null   object
 5   Método de Pagamento  7421 non-null   object
 6   Localização          6735 non-null   object
 7   Data da Transação    9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


* Verificação de nulos

In [ ]:
df.isna().sum()

,0
ID da Transação,0
Item,333
Quantidade,138
Preço Unitário,179
Total Gasto,173
Método de Pagamento,2579
Localização,3265
Data da Transação,159


* Verificação de valores unicos por colunas

In [ ]:
df.nunique()

,0
ID da Transação,10000
Item,10
Quantidade,7
Preço Unitário,8
Total Gasto,19
Método de Pagamento,5
Localização,4
Data da Transação,367


In [ ]:
df.describe()

,ID da Transação,Item,Quantidade,Preço Unitário,Total Gasto,Método de Pagamento,Localização,Data da Transação
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [ ]:
for i in df.columns:
    display(df['Método de Pagamento'].value_counts())

,count
Método de Pagamento,
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


,count
Método de Pagamento,
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


,count
Método de Pagamento,
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


,count
Método de Pagamento,
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


,count
Método de Pagamento,
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


,count
Método de Pagamento,
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


,count
Método de Pagamento,
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


,count
Método de Pagamento,
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


##Tratamento dos dados


In [ ]:
# Forçar a conversão das colunas para números (o que for erro vira NaN)
colunas_numericas = ['Quantidade', 'Preço Unitário', 'Total Gasto']
for col in colunas_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# Aplicar regra de três vias para preencher os buracos (NaN)
df['Total Gasto'] = df['Total Gasto'].fillna(df['Quantidade'] * df['Preço Unitário'])
df['Preço Unitário'] = df['Preço Unitário'].fillna(df['Total Gasto'] / df['Quantidade'])
df['Quantidade'] = df['Quantidade'].fillna(df['Total Gasto'] / df['Preço Unitário'])

In [ ]:
df[colunas_numericas].isna().sum()

,0
Quantidade,38
Preço Unitário,38
Total Gasto,40


In [ ]:
# Filtrar as linhas que possuem algum valor vazio (NaN) nas colunas numéricas
linhas_vazias = df[df[colunas_numericas].isnull().any(axis=1)]

In [ ]:
# Contagem dos valores baseado no nome dos items
display(linhas_vazias['Item'].value_counts())

,count
Item,
Sandwich,9
Tea,8
Cookie,8
Coffee,7
Smoothie,7
Cake,5
Salad,4
Juice,4
UNKNOWN,2


In [ ]:
# Pegar apenas as colunas de Item e Preço, remover os vazios e mostrar os valores únicos
tabela_de_precos = df[['Item', 'Preço Unitário']].dropna().drop_duplicates()
display(tabela_de_precos.sort_values(by='Item'))

,Item,Preço Unitário
1,Cake,3.0
0,Coffee,2.0
2,Cookie,1.0
283,ERROR,1.0
14,ERROR,1.5
281,ERROR,2.0
216,ERROR,4.0
69,ERROR,3.0
88,ERROR,5.0
17,Juice,3.0


In [ ]:
# Criar uma tabela relacionando o Item ao Preço Unitário
tabela_de_precos = df[['Item', 'Preço Unitário']].dropna().drop_duplicates()
dicionario_precos = dict(zip(tabela_de_precos['Item'], tabela_de_precos['Preço Unitário']))

In [ ]:
#  Usar a tabela para preencher os preços unitários que estavam faltando (NaN)
df['Preço Unitário'] = df['Preço Unitário'].fillna(df['Item'].map(dicionario_precos))

In [ ]:
# Aplicar regra para preencher os buracos (NaN) restantes
df['Total Gasto'] = df['Total Gasto'].fillna(df['Quantidade'] * df['Preço Unitário'])
df['Quantidade'] = df['Quantidade'].fillna(df['Total Gasto'] / df['Preço Unitário'])

In [ ]:
# Remover linhas que têm NaN nas colunas numéricas
df = df.dropna(subset=colunas_numericas)

In [ ]:
# Transformar 'UNKNOWN' e 'ERROR' da coluna Item em valores nulos (NaN)
df['Método de Pagamento'] = df['Método de Pagamento'].replace(['UNKNOWN','ERROR'], pd.NA)

In [ ]:
# Preencher todos os vazios (NaN) com a nossa nova categoria
df['Método de Pagamento'] = df['Método de Pagamento'].fillna('Sem Método')

In [ ]:
# Transformar 'UNKNOWN' e 'ERROR' da coluna Item em valores nulos (NaN)
df['Item'] = df['Item'].replace(['UNKNOWN', 'ERROR'], pd.NA)

In [ ]:
# Criar um cardápio reverso (Preço -> Item) usando o dicionário que já tínhamos
# Se você não tiver o dicionario_precos na memória, recriamos assim:
tabela_de_precos = df[['Item', 'Preço Unitário']].dropna().drop_duplicates()
dicionario_reverso = dict(zip(tabela_de_precos['Preço Unitário'], tabela_de_precos['Item']))

In [ ]:
# Preencher os itens vazios baseados no Preço Unitário
df['Item'] = df['Item'].fillna(df['Preço Unitário'].map(dicionario_reverso))

In [ ]:
# O 'coerce' faz com que datas inválidas ou erros virem NaT (Not a Time, o NaN das datas)
df['Data da Transação'] = pd.to_datetime(df['Data da Transação'], errors='coerce')

In [ ]:
# Substituir "UNKNOWN" e "ERROR" por valores nulos (NaN)
df['Localização'] = df['Localização'].replace(['UNKNOWN', 'ERROR'], pd.NA)

In [ ]:
# Preencher todos os vazios (NaN) com a nossa nova categoria
df['Localização'] = df['Localização'].fillna('Sem Localização')

In [ ]:
df.duplicated().sum()

np.int64(0)

## analise descritiva

1. Qual foi o faturamento total da cafeteria?

In [ ]:
print('Aproximadamente:')
print(f'R$ {df['Total Gasto'].sum():,.2f}')

Aproximadamente:
R$ 89,086.00


2. Qual localização teve maior faturamento?

In [ ]:
localizacao_fat =  df.groupby('Localização')['Total Gasto'].sum().reset_index().sort_values(by='Total Gasto',ascending=False)
fig = px.pie(
    localizacao_fat,
    names='Localização',
    values= 'Total Gasto',
    height=500, width=500,
    color_discrete_sequence= px.colors.carto.Redor_r,
    title='Faturamento por Loja'
)

fig.update_traces(pull=[0.1],hole=0.4)

fig.show()

3. Qual item foi o mais vendido em quantidade?

In [ ]:
prod_qtd = df.groupby('Item')['Quantidade'].sum().sort_values( ascending=False).reset_index()
fig = px.bar(
    prod_qtd,
    x='Item',
    y='Quantidade',
    text=prod_qtd['Quantidade'],
    title='Vendas por produtos em quantidade',
    color='Item',
    color_discrete_sequence= px.colors.carto.Redor_r
)

fig.update_traces(
    textposition='outside',
    texttemplate='%{text:.3s}'
)

fig.show()

4. Qual item gerou maior receita?

In [ ]:
prod_rec = df.groupby('Item')['Total Gasto'].sum().sort_values(ascending=False).reset_index().head()

fig = px.bar(
    prod_rec,
    x = 'Item',
    y='Total Gasto',
    text=prod_rec['Total Gasto'],
    color='Total Gasto',
    color_continuous_scale= px.colors.carto.Redor,
    title='Receita por produto'
)

fig.update_traces(textposition='outside', texttemplate='$ %{text:0.2s}')

fig.show()

5. Qual método de pagamento foi mais utilizado?

In [ ]:
pagamentos = df['Método de Pagamento'].value_counts().reset_index()

fig = px.pie(
    pagamentos, names='Método de Pagamento',
    values='count',
    height=500, width=500,
    color_discrete_sequence=px.colors.carto.Redor_r,
    title='Métodos de pagamentos mais utlizados'
)

# 'label+value' mostra o nome do método e a quantidade juntos
fig.update_traces(textinfo='value',hole=0.45)

fig.show()

6. Calcular o faturamento mensal.

In [ ]:
df['Data da Transação'] = df['Data da Transação'].dt.to_period('M')

In [ ]:
fat_mes = df.groupby('Data da Transação')['Total Gasto'].sum().reset_index()
fat_mes['Data da Transação'] = fat_mes['Data da Transação'].dt.to_timestamp()
fig = px.bar(
    fat_mes,
    x='Data da Transação',
    y='Total Gasto',
    title='Vendas Mensais',
    text=fat_mes['Total Gasto'],
    color_discrete_sequence=px.colors.carto.Redor_r
)

fig.update_traces(textposition='outside', texttemplate='$ %{text:0.2s}')

fig.show()

6. Calcular o faturamento mensal por loja

In [ ]:
fat_mes_lj = df.groupby(['Data da Transação','Localização'])['Total Gasto'].sum().reset_index()
fat_mes_lj['Data da Transação'] = fat_mes_lj['Data da Transação'].dt.to_timestamp()
fig = px.line(
    fat_mes_lj,
    x='Data da Transação',
    y='Total Gasto',
    markers=True,
    title='Comparação de Vendas Mensais por Loja',
    color='Localização',

)

fig.show()

7. Os produtos mais vendidos são os mesmos nas duas localizações?

In [ ]:
from pandas.core.indexes import category
prod_qtd = df.groupby(['Item','Localização'])['Quantidade'].sum().sort_values( ascending=[False]).reset_index()
fig = px.bar(
    prod_qtd,
    x='Item',
    y='Quantidade',
    text=prod_qtd['Quantidade'],
    title='Vendas por produtos em quantidade',
    color='Item',
    color_discrete_sequence= px.colors.carto.Redor_r,
    facet_col='Localização',
    barmode='stack'
)

fig.update_traces(
    textposition='outside',
    texttemplate='%{text:.4m}'
)
fig.update_yaxes()

# Organiza os paineis internos na ordem decrescente
fig.update_xaxes(matches=None,categoryorder='total descending')

fig.show()

8. Escreva uma pequena interpretação dos resultados encontrados.

Podemos ver que um dos maiores problemas da base é a poluição dela. Mesmo com a correção de boa parte dos dados, questões como localização e datas vêm com um grande volume de problemas. Conseguimos ver que nas lojas os produtos mais vendidos são os mesmos; porém, se acompanharmos as métricas dos dados sem localização, temos uma mudança bem interessante no top 3 com o aparecimento do café. Também podemos notar que, quando buscamos o faturamento por loja, a categoria "Sem Localização" mostra o maior faturamento, chegando a quase 40% do faturamento total. O mesmo ocorre com os métodos de pagamento, em que a maior parte é classificada como "Sem Método". Fica a pergunta para vocês: o que sugeririam para o dono do negócio para que não houvesse mais esse tipo de perda de informação?

9. analise vendas dias da semana


In [ ]:
df['Dia Semana'] = df['Data da Transação'].dt.dayofweek
df['Dia Semana'] = df['Dia Semana'].map({
    6:'Domingo',
    0:'Segunda',
    1:'Terça',
    2:'Quarta',
    3:'Quinta',
    4:'Sexta',
    5:'Sábado'
})
df[''] = df['Dia Semana'].fillna('Sem Data')

In [ ]:
analise_semanal = df.groupby('Dia Semana')['Total Gasto'].sum().reset_index().sort_values(by='Total Gasto', ascending=False)

fig = px.bar(
    analise_semanal,
    x='Dia Semana',
    y='Total Gasto',
    text=analise_semanal['Total Gasto'],
    color='Total Gasto',
    color_continuous_scale= px.colors.carto.Redor
)

fig.update_traces(textposition='outside', texttemplate='$ %{text:.2s}')

fig.show()

In [ ]:
ordem_dias = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']

analise_semanal_prod = df.groupby(['Dia Semana','Item'])['Quantidade'].sum().reset_index()
analise_semanal_prod['Dia Semana'] = pd.Categorical(
    analise_semanal_prod['Dia Semana'],
    categories=ordem_dias,
    ordered=True
)

analise_semanal_prod = analise_semanal_prod.sort_values('Dia Semana')

fig = px.line(
    analise_semanal_prod,
    x='Dia Semana',
    y='Quantidade',
    color='Item',
    markers=True
)

fig.show()

10. calculo do ticket meédio

In [ ]:
# Calculando o Ticket Médio usando a função que você descobriu
ticket_medio_dia = df.groupby('Dia Semana')['Total Gasto'].mean().reset_index()

# Renomeando a coluna para o gráfico ficar mais elegante
ticket_medio_dia.rename(columns={'Total Gasto': 'Ticket Médio'}, inplace=True)

# Criando o gráfico de barras
fig = px.bar(
    ticket_medio_dia,
    x='Dia Semana',
    y='Ticket Médio',
    category_orders={'Dia Semana': ordem_dias}, # Mantendo a nossa ordem cronológica
    title='Ticket Médio por Dia da Semana',
    text_auto='.2f',
    color='Ticket Médio',
    color_continuous_scale= px.colors.carto.Redor
)

fig.update_traces(textposition='outside')

fig.show()